In [1]:
# -*-coding:utf-8-*-
"""
利用 Qwen 3.5 进行文本分类任务（Kaggle ）
"""
from rich import print
from rich.console import Console
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# 替换后的分类类别和样例（旅游场景文本分类）
class_examples = {
    '景点介绍': '故宫博物院位于北京中轴线中心，是明清两代的皇家宫殿，拥有70多座宫殿、9000余间房屋，馆藏文物超186万件，是世界文化遗产之一。',
    '旅游攻略': '去云南大理旅游建议安排5天行程：Day1逛大理古城，Day2环洱海骑行，Day3爬苍山，Day4去双廊古镇，Day5体验喜洲扎染，住宿优先选洱海边的民宿。',
    '游客评价': '这次去桂林阳朔的遇龙河竹筏漂流体验超棒！筏工师傅很热情，沿途山水风景如画，就是旺季人有点多，建议错峰出行。',
    '酒店预订': '我想预订三亚亚龙湾万豪度假酒店的海景大床房，入住时间是下周五，退房时间是下周日，需要含双早的房型，麻烦确认是否有房。'
}

def init_prompts():
    '''
    初始化前置prompt，便于模型做Few-shot
    :return: dict字典
    '''
    class_list = list(class_examples.keys())
    print(f'分类的类别数：{len(class_list)}，类别：{class_list}')

    # Qwen 采用 messages 格式构建对话历史
    messages = [
        {"role": "system", "content": f'现在你是一个文本分类器，你需要按照要求将我给你的句子分类到：{class_list}类别中。只输出类别名称，不要额外解释。'},
        {"role": "user", "content": f'"{class_examples["景点介绍"]}"是{class_list}里的什么类别'},
        {"role": "assistant", "content": "景点介绍"},
        {"role": "user", "content": f'"{class_examples["旅游攻略"]}"是{class_list}里的什么类别'},
        {"role": "assistant", "content": "旅游攻略"},
        {"role": "user", "content": f'"{class_examples["游客评价"]}"是{class_list}里的什么类别'},
        {"role": "assistant", "content": "游客评价"},
        {"role": "user", "content": f'"{class_examples["酒店预订"]}"是{class_list}里的什么类别'},
        {"role": "assistant", "content": "酒店预订"},
    ]

    return {"class_list": class_list, "messages": messages}


def inference(sentences: list, custom_settings: dict):
    """
    推理函数。
    Args:
        sentences (List[str]): 待推理的句子。
        custom_settings (dict): 初始设定，包含人为给定的 few-shot example。
    """
    for sentence in sentences:
        with console.status("[bold bright_green] Model Inference..."):
            # 复制历史，避免修改全局messages
            current_messages = custom_settings["messages"].copy()
            # 添加当前用户问题
            current_messages.append({
                "role": "user",
                "content": f'"{sentence}"是{custom_settings["class_list"]}里的什么类别？只输出类别名称，不要额外解释。'
            })
            # 构建模型输入
            text = tokenizer.apply_chat_template(
                current_messages,
                tokenize=False,
                add_generation_prompt=True
            )
            model_inputs = tokenizer([text], return_tensors="pt").to(device)

            # 生成回答
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=64,
                temperature=0.1,
                do_sample=False
            )
            # 只保留生成的部分，去掉输入的prompt
            generated_ids = [
                output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
            ]
            response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        print(f'>>> [bold bright_red]sentence: {sentence}')
        print(f'>>> [bold bright_green]inference answer: {response.strip()}')
        print("*" * 80)


if __name__ == '__main__':
    console = Console()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'使用设备: {device}')

    
    # 1) 从 Hugging Face Hub 加载（需要联网）
    # 2) 或上传到 Kaggle Dataset 的权重加载
    model_name_or_path = "Qwen/Qwen2.5-7B-Instruct" 
    # model_name_or_path = "/kaggle/input/your-qwen-model-folder"

    tokenizer = AutoTokenizer.from_pretrained(
        model_name_or_path,
        trust_remote_code=True
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name_or_path,
        torch_dtype="auto",
        device_map="auto",
        trust_remote_code=True
    )
    model = model.eval()

    # 替换后的测试句子（旅游场景）
    sentences = [
        "张家界国家森林公园以峰林、峡谷、溶洞景观著称，是首批国家5A级旅游景区，也是电影《阿凡达》的取景地之一。",
        "想订成都春熙路附近的四星级酒店，入住时间是国庆期间，需要有停车场和洗衣服务的房间。",
        "西安旅游避坑指南：兵马俑景区门口的讲解不要随便找，建议提前在官方公众号预约正规讲解。"
    ]
    custom_settings = init_prompts()
    print(custom_settings)

    inference(sentences, custom_settings)

使用设备: cuda

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

********************************************************************************

Output()

>>> sentence: 西安旅游避坑指南：兵马俑景区门口的讲解不要随便找，建议提前在官方公众号预约正规讲解。

>>> inference answer: 旅游攻略

********************************************************************************